In [1]:
import geopandas as gpd
import pandas as pd
import numpy as np
import requests
import io
from shapely.geometry import box

gpkg = r"C:\Users\deqin\Downloads\matcad2022_de-be_bfp_mps1.gpkg"

gdf = gpd.read_file(
    gpkg,
    layer="Mat1"
)

print("MATCAD buildings:", len(gdf))
print("CRS:", gdf.crs)

MATCAD buildings: 534612
CRS: EPSG:25833


In [2]:
energy_wfs = (
    "https://gdi.berlin.de/services/wfs/"
    "gebaeudeeignung_waerme"
)

energy_layer = (
    "gebaeudeeignung_waerme:"
    "a_waermeversorg_gebaeudedaten_2022"
)

xmin, ymin, xmax, ymax = gdf.total_bounds

params = {
    "service": "WFS",
    "version": "2.0.0",
    "request": "GetFeature",
    "typeNames": energy_layer,
    "outputFormat": "application/json",
    "srsName": "EPSG:25833",
    "bbox": f"{xmin},{ymin},{xmax},{ymax},EPSG:25833"
}

response = requests.get(
    energy_wfs,
    params=params,
    timeout=300
)

response.raise_for_status()

energy_gdf = gpd.read_file(
    io.BytesIO(response.content)
)

print("Energy buildings:", len(energy_gdf))
print("CRS:", energy_gdf.crs)

Energy buildings: 537198
CRS: EPSG:25833


In [3]:
joined = gpd.sjoin(
    gdf,
    energy_gdf,
    how="left",
    predicate="intersects"
)

print("Joined rows:", len(joined))

Joined rows: 930408


In [4]:
# Energy geometry über index_right wieder anhängen

energy_geom = energy_gdf.geometry.rename(
    "energy_geometry"
)

joined = joined.join(
    energy_geom,
    on="index_right"
)

# Schnittfläche berechnen
joined["intersection_area_m2"] = (
    joined.geometry
    .intersection(joined["energy_geometry"])
    .area
)

# MATCAD-Fläche
joined["matcad_area_m2"] = (
    joined.geometry.area
)

# Überlappungsanteil
joined["overlap_ratio_matcad"] = (
    joined["intersection_area_m2"]
    / joined["matcad_area_m2"]
)

# Pro MATCAD-Gebäude nur den Match mit
# größtem Überlappungsanteil behalten

joined = (
    joined
    .sort_values(
        "overlap_ratio_matcad",
        ascending=False
    )
    .drop_duplicates(
        subset="bldg_gmlid",
        keep="first"
    )
    .copy()
)

print("Rows after best-overlap selection:", len(joined))

C:\Users\deqin\miniconda3\envs\geo_env\lib\site-packages\shapely\set_operations.py:131: RuntimeWarning: invalid value encountered in intersection
  return lib.intersection(a, b, **kwargs)


Rows after best-overlap selection: 534612


In [6]:
joined["nwbz"] = pd.to_numeric(
    joined["nwbz"],
    errors="coerce"
)

positive_energy = joined[
    joined["nwbz"] > 0
].copy()

print("Positive energy buildings:", len(positive_energy))

Positive energy buildings: 335381


In [7]:
bounds = positive_energy.geometry.bounds

positive_energy["bbox_length_m"] = (
    bounds["maxx"] - bounds["minx"]
)

positive_energy["bbox_width_m"] = (
    bounds["maxy"] - bounds["miny"]
)

positive_energy = positive_energy[
    (positive_energy["bbox_length_m"] >= 3) &
    (positive_energy["bbox_width_m"] >= 3)
].copy()

print("After geometry filter:", len(positive_energy))

After geometry filter: 332352


In [8]:
positive_energy["footprint_area_m2"] = (
    positive_energy.geometry.area
)

positive_energy["height_proxy_m"] = (
    positive_energy["bldg_volume"]
    / positive_energy["footprint_area_m2"]
)

print(
    positive_energy["height_proxy_m"].describe()
)

count    332352.000000
mean          9.906565
std           6.141872
min           0.002619
25%           6.056944
50%           7.836929
75%          12.681011
max          95.079630
Name: height_proxy_m, dtype: float64


In [9]:
positive_energy = positive_energy[
    (positive_energy["height_proxy_m"] >= 2) &
    (positive_energy["height_proxy_m"] <= 30)
].copy()

print("After height filter:", len(positive_energy))

After height filter: 327339


In [10]:
positive_energy["energy_intensity_kwh_m3a"] = (
    positive_energy["nwbz"]
    / positive_energy["bldg_volume"]
)

print(
    positive_energy[
        "energy_intensity_kwh_m3a"
    ].describe()
)

count    327339.000000
mean         20.543224
std          66.167417
min           0.740552
25%          14.765169
50%          18.707457
75%          22.906141
max       21127.113604
Name: energy_intensity_kwh_m3a, dtype: float64


In [11]:
positive_energy = positive_energy[
    positive_energy["energy_intensity_kwh_m3a"] <= 500
].copy()

print(
    "After energy-intensity filter:",
    len(positive_energy)
)

After energy-intensity filter: 327295


In [12]:
denkmal_wfs_url = (
    "https://gdi.berlin.de/services/wfs/denkmale"
)

denkmal_layer = "denkmale:denkmale"

xmin, ymin, xmax, ymax = gdf.total_bounds

params = {
    "service": "WFS",
    "version": "2.0.0",
    "request": "GetFeature",
    "typeNames": denkmal_layer,
    "outputFormat": "application/json",
    "srsName": "EPSG:25833",
    "bbox": f"{xmin},{ymin},{xmax},{ymax},EPSG:25833"
}

response = requests.get(
    denkmal_wfs_url,
    params=params,
    timeout=300
)

response.raise_for_status()

denkmal_gdf = gpd.read_file(
    io.BytesIO(response.content)
)

print("Monuments:", len(denkmal_gdf))

Monuments: 9577


In [13]:
denkmal_join = gpd.sjoin(
    positive_energy[
        ["bldg_gmlid", "geometry"]
    ],
    denkmal_gdf[
        ["geometry"]
    ],
    how="left",
    predicate="intersects"
)

denkmal_ids = set(
    denkmal_join.loc[
        denkmal_join["index_right"].notna(),
        "bldg_gmlid"
    ]
)

positive_energy["denkmal_status"] = (
    positive_energy["bldg_gmlid"]
    .isin(denkmal_ids)
)

print(
    positive_energy["denkmal_status"]
    .value_counts()
)

denkmal_status
False    291472
True      35823
Name: count, dtype: int64


In [14]:
model_data = positive_energy.copy()

# Temporäre technische Spalten entfernen
drop_cols = [
    "index_right",
    "energy_geometry",
    "intersection_area_m2",
    "matcad_area_m2"
]

model_data = model_data.drop(
    columns=[
        c for c in drop_cols
        if c in model_data.columns
    ]
)

print("Final model_data shape:", model_data.shape)

display(
    model_data.head()
)

Final model_data shape: (327295, 67)


,bldg_gmlid,AGS,bldg_area,bldg_volume,bldg_function,BtySetID,BtyID,BtySig,Mat_Sum,Mat_ConNrm,...,typ_mod1,k2_w_l,nwbz,overlap_ratio_matcad,bbox_length_m,bbox_width_m,footprint_area_m2,height_proxy_m,energy_intensity_kwh_m3a,denkmal_status
522949,DEBE10YYW00002xf,11000000,10434.53,78359.13,31001_2054,4.0,157,TRD,35189.650,17903.029,...,Nichtwohngebäude,K01 - beheizt,1438202.0,1.0,122.847,135.610,10434.386454,7.509702,18.353981,False
415594,DEBE09YYP0003eLR,11000000,836.77,13304.39,31001_1010,4.0,246,MFH,6182.778,2697.168,...,Wohngebäude - zentrale Versorgung,K10 - Wohngebäude inkl. Mischnutzungen (nachri...,231117.0,1.0,77.907,11.688,836.761827,15.899853,17.371484,False
12175,DEBE05YYR0000Iro,11000000,7.38,17.86,31001_1000,4.0,159,NAS,14.562,9.922,...,Wohngebäude - dezentrale Versorgung,K10 - Wohngebäude inkl. Mischnutzungen (nachri...,547.0,1.0,4.295,3.706,7.380064,2.420033,30.627100,True
427623,DEBE10YYX00004Df,11000000,8.19,24.45,31001_1010,4.0,159,NAS,19.938,13.584,...,Wohngebäude - dezentrale Versorgung,K10 - Wohngebäude inkl. Mischnutzungen (nachri...,9728.0,1.0,5.579,3.481,8.189711,2.985453,397.873211,False
348181,DEBE09YYP0000DVt,11000000,6.19,26.43,31001_1010,4.0,159,NAS,21.550,14.684,...,Wohngebäude - dezentrale Versorgung,K10 - Wohngebäude inkl. Mischnutzungen (nachri...,547.0,1.0,3.613,3.508,6.194290,4.266833,20.696179,False


In [15]:
output_csv = (
    r"C:\Users\deqin\Downloads"
    r"\MATCAD_Energy_Model_Data.csv"
)

# Geometrie für CSV entfernen
model_data_csv = model_data.drop(
    columns="geometry"
).copy()

model_data_csv.to_csv(
    output_csv,
    index=False
)

print("CSV saved successfully:")
print(output_csv)

print("Rows:", len(model_data_csv))
print("Columns:", len(model_data_csv.columns))


CSV saved successfully:
C:\Users\deqin\Downloads\MATCAD_Energy_Model_Data.csv
Rows: 327295
Columns: 66


In [54]:
baualter_wfs_url = (
    "https://gdi.berlin.de/services/wfs/"
    "ua_gebaeudealter"
)

print("Gebäudealter-WFS vorbereitet")

Gebäudealter-WFS vorbereitet


In [56]:
import requests
from xml.etree import ElementTree as ET

capabilities_url = (
    baualter_wfs_url
    + "?service=WFS&version=2.0.0&request=GetCapabilities"
)

response = requests.get(capabilities_url, timeout=60)

print("HTTP Status:", response.status_code)
print(response.text[:1000])

HTTP Status: 200
<?xml version="1.0" encoding="UTF-8"?><wfs:WFS_Capabilities version="2.0.0" xmlns:xsi="http://www.w3.org/2001/XMLSchema-instance" xmlns="http://www.opengis.net/wfs/2.0" xmlns:wfs="http://www.opengis.net/wfs/2.0" xmlns:ows="http://www.opengis.net/ows/1.1" xmlns:gml="http://www.opengis.net/gml/3.2" xmlns:fes="http://www.opengis.net/fes/2.0" xmlns:xlink="http://www.w3.org/1999/xlink" xmlns:xs="http://www.w3.org/2001/XMLSchema" xsi:schemaLocation="http://www.opengis.net/wfs/2.0 http://schemas.opengis.net/wfs/2.0/wfs.xsd http://inspire.ec.europa.eu/schemas/inspire_dls/1.0 https://inspire.ec.europa.eu/schemas/inspire_dls/1.0/inspire_dls.xsd" xmlns:xml="http://www.w3.org/XML/1998/namespace" xmlns:inspire_dls="http://inspire.ec.europa.eu/schemas/inspire_dls/1.0" xmlns:inspire_common="http://inspire.ec.europa.eu/schemas/common/1.0" xmlns:ua_gebaeudealter="ua_gebaeudealter" updateSequence="541242"><ows:ServiceIdentification><ows:Title>Gebäudealter der Wohnbebauung (Umweltatlas)<

In [19]:
root = ET.fromstring(response.content)

namespaces = {
    "wfs": "http://www.opengis.net/wfs/2.0",
    "ows": "http://www.opengis.net/ows/1.1"
}

layer_names = []

for elem in root.findall(".//wfs:FeatureType/wfs:Name", namespaces):
    layer_names.append(elem.text)

print("Verfügbare Layer:")
for name in layer_names:
    print(name)

Verfügbare Layer:
ua_gebaeudealter:ua_gebaeudealter_baualter


In [ ]:
print("HTTP Status:", response.status_code)
print(response.text[:1000])

In [21]:
from xml.etree import ElementTree as ET

root = ET.fromstring(response.content)

namespaces = {
    "wfs": "http://www.opengis.net/wfs/2.0"
}

layer_names = []

for elem in root.findall(".//wfs:FeatureType/wfs:Name", namespaces):
    layer_names.append(elem.text)

print("Verfügbare Layer:")
for name in layer_names:
    print(name)

Verfügbare Layer:
ua_gebaeudealter:ua_gebaeudealter_baualter


In [25]:
feature_url = (
    baualter_wfs_url
    + "?service=WFS"
    + "&version=2.0.0"
    + "&request=GetFeature"
    + "&typeNames=ua_gebaeudealter:ua_gebaeudealter_baualte"
    + "&outputFormat=application/json"
    + "&count=1"
)

feature_response = requests.get(feature_url, timeout=60)

print("HTTP Status:", feature_response.status_code)
print(feature_response.text[:2000])

HTTP Status: 400
<?xml version="1.0" encoding="UTF-8"?><ows:ExceptionReport xmlns:xs="http://www.w3.org/2001/XMLSchema" xmlns:ows="http://www.opengis.net/ows/1.1" xmlns:xsi="http://www.w3.org/2001/XMLSchema-instance" version="2.0.0" xsi:schemaLocation="http://www.opengis.net/ows/1.1 http://10.10.212.83/geoserver/schemas/ows/1.1.0/owsAll.xsd">
  <ows:Exception exceptionCode="InvalidParameterValue" locator="typeName">
    <ows:ExceptionText>Feature type ua_gebaeudealter:ua_gebaeudealter_baualte unknown</ows:ExceptionText>
  </ows:Exception>
</ows:ExceptionReport>



In [26]:
feature_url = (
    baualter_wfs_url
    + "?service=WFS"
    + "&version=2.0.0"
    + "&request=GetFeature"
    + "&typeName=ua_gebaeudealter_baualte"
    + "&outputFormat=application/json"
    + "&count=1"
)

feature_response = requests.get(feature_url, timeout=60)

print("HTTP Status:", feature_response.status_code)
print(feature_response.text[:3000])

HTTP Status: 400
<?xml version="1.0" encoding="UTF-8"?><ows:ExceptionReport xmlns:xs="http://www.w3.org/2001/XMLSchema" xmlns:ows="http://www.opengis.net/ows/1.1" xmlns:xsi="http://www.w3.org/2001/XMLSchema-instance" version="2.0.0" xsi:schemaLocation="http://www.opengis.net/ows/1.1 http://10.10.212.83/geoserver/schemas/ows/1.1.0/owsAll.xsd">
  <ows:Exception exceptionCode="NoApplicableCode">
    <ows:ExceptionText>WFS 2.0 requires typeNames, not typeName</ows:ExceptionText>
  </ows:Exception>
</ows:ExceptionReport>



In [27]:
feature_url = (
    baualter_wfs_url
    + "?service=WFS"
    + "&version=2.0.0"
    + "&request=GetFeature"
    + "&typeNames=ua_gebaeudealter_baualte"
    + "&outputFormat=application/json"
    + "&count=1"
)

feature_response = requests.get(feature_url, timeout=60)

print("HTTP Status:", feature_response.status_code)
print(feature_response.text[:3000])

HTTP Status: 400
<?xml version="1.0" encoding="UTF-8"?><ows:ExceptionReport xmlns:xs="http://www.w3.org/2001/XMLSchema" xmlns:ows="http://www.opengis.net/ows/1.1" xmlns:xsi="http://www.w3.org/2001/XMLSchema-instance" version="2.0.0" xsi:schemaLocation="http://www.opengis.net/ows/1.1 http://10.10.212.83/geoserver/schemas/ows/1.1.0/owsAll.xsd">
  <ows:Exception exceptionCode="InvalidParameterValue" locator="typeName">
    <ows:ExceptionText>Feature type :ua_gebaeudealter_baualte unknown</ows:ExceptionText>
  </ows:Exception>
</ows:ExceptionReport>



In [57]:
# Zeigt den vollständigen FeatureType-Abschnitt aus den Capabilities

for feature_type in root.findall(".//{http://www.opengis.net/wfs/2.0}FeatureType"):
    print(ET.tostring(feature_type, encoding="unicode")[:5000])

<ns0:FeatureType xmlns:ns0="http://www.opengis.net/wfs/2.0" xmlns:ns1="http://www.opengis.net/ows/1.1" xmlns:ns2="http://www.w3.org/1999/xlink"><ns0:Name>ua_gebaeudealter:ua_gebaeudealter_baualter</ns0:Name><ns0:Title>Gebäudealter der Wohngebäude 2016</ns0:Title><ns0:Abstract>Dargestellt werden die überwiegenden Baualtersklassen (Angaben in Dekaden) der Wohngebäude auf der Ebene der Block- und Blockteilflächen der Grundkarte 1 : 5.000 (ISU5, Raumbezug Umweltatlas 2010).</ns0:Abstract><ns1:Keywords><ns1:Keyword>Gebäudealter</ns1:Keyword><ns1:Keyword>jüngstes</ns1:Keyword><ns1:Keyword>Berlin</ns1:Keyword><ns1:Keyword>WMS</ns1:Keyword><ns1:Keyword>Baugeschichte</ns1:Keyword><ns1:Keyword>Gebäudeart</ns1:Keyword><ns1:Keyword>Baualter</ns1:Keyword><ns1:Keyword>infoMapAccessService</ns1:Keyword><ns1:Keyword>überwiegend</ns1:Keyword><ns1:Keyword>Umweltatlas</ns1:Keyword><ns1:Keyword>Wohngebäude</ns1:Keyword><ns1:Keyword>Wohnnutzung</ns1:Keyword><ns1:Keyword>Stadtgeschichte</ns1:Keyword><ns1:Ke

In [29]:
age_layer = "ua_gebaeudealter:ua_gebaeudealter_baualter"

feature_url = (
    baualter_wfs_url
    + "?service=WFS"
    + "&version=2.0.0"
    + "&request=GetFeature"
    + "&typeNames=" + age_layer
    + "&outputFormat=application/json"
    + "&count=1"
)

feature_response = requests.get(feature_url, timeout=60)

print("HTTP Status:", feature_response.status_code)
print(feature_response.text[:3000])

HTTP Status: 200
{"type":"FeatureCollection","features":[{"type":"Feature","id":"ua_gebaeudealter_baualter.0100980071000100","geometry":{"type":"Polygon","coordinates":[[[389747.7595,5821227.6169],[389770.4615,5821239.0615],[389828.0301,5821229.4851],[389865.2303,5821250.6119],[389817.0262,5821340.3046],[389813.8815,5821341.1044],[389811.0477,5821341.9383],[389714.2588,5821285.3846],[389747.7595,5821227.6169]]]},"geometry_name":"geom","properties":{"schluessel":"0100980071000100","freistehen":null,"anderertyp":null,"doppelhaus":null,"gereihtes":3,"x_bis_1900":null,"x1901_1910":null,"x1911_1920":null,"x1921_1930":null,"x1931_1940":null,"x1941_1950":null,"x1951_1960":null,"x1961_1970":null,"x1971_1980":3,"x1981_1990":null,"x1991_2000":null,"x2001_2010":null,"x2011_2015":null,"ueberw_dekade_woh_neu":"1971-1980","ew2015":267,"typ":9,"typklar":"Großsiedlungen und Punkthochhäuser (1960er-1980er), 4–11-geschossig"},"bbox":[389714.2588,5821227.6169,389865.2303,5821341.9383]}],"totalFeatures":1

In [58]:
import json
from io import BytesIO

age_gdf = gpd.read_file(
    BytesIO(feature_response.content)
)

print("Gebäudealter-Flächen:", len(age_gdf))
print("CRS:", age_gdf.crs)
print("Spalten:")
print(age_gdf.columns.tolist())

Gebäudealter-Flächen: 1
CRS: EPSG:25833
Spalten:
['id', 'schluessel', 'freistehen', 'anderertyp', 'doppelhaus', 'gereihtes', 'x_bis_1900', 'x1901_1910', 'x1911_1920', 'x1921_1930', 'x1931_1940', 'x1941_1950', 'x1951_1960', 'x1961_1970', 'x1971_1980', 'x1981_1990', 'x1991_2000', 'x2001_2010', 'x2011_2015', 'ueberw_dekade_woh_neu', 'ew2015', 'typ', 'typklar', 'geometry']


In [59]:
# Überblick über den geladenen Gebäudealter-Datensatz

print("Anzahl Gebäudealter-Flächen:", len(age_gdf))
print("CRS:", age_gdf.crs)

print("\nDatentypen:")
print(age_gdf.dtypes)

print("\nVerteilung der dominanten Baualtersklassen:")
print(age_gdf["ueberw_dekade_woh_neu"].value_counts(dropna=False))

print("\nBeispielwerte:")
display(
    age_gdf[
        ["ueberw_dekade_woh_neu", "typ", "typklar", "ew2015"]
    ].head(10)
)

Anzahl Gebäudealter-Flächen: 1
CRS: EPSG:25833

Datentypen:
id                         object
schluessel                 object
freistehen                 object
anderertyp                 object
doppelhaus                 object
gereihtes                   int32
x_bis_1900                 object
x1901_1910                 object
x1911_1920                 object
x1921_1930                 object
x1931_1940                 object
x1941_1950                 object
x1951_1960                 object
x1961_1970                 object
x1971_1980                  int32
x1981_1990                 object
x1991_2000                 object
x2001_2010                 object
x2011_2015                 object
ueberw_dekade_woh_neu      object
ew2015                      int32
typ                         int32
typklar                    object
geometry                 geometry
dtype: object

Verteilung der dominanten Baualtersklassen:
ueberw_dekade_woh_neu
1971-1980    1
Name: count, dtype: int64

B

,ueberw_dekade_woh_neu,typ,typklar,ew2015
0,1971-1980,9,Großsiedlungen und Punkthochhäuser (1960er-198...,267


In [32]:
import json

print("HTTP Status:", feature_response.status_code)
print("Content-Type:", feature_response.headers.get("Content-Type"))

age_json = feature_response.json()

print("Top-Level-Schlüssel:")
print(age_json.keys())

print("\nTyp des JSON-Inhalts:")
print(age_json.get("type"))

print("\nAnzahl Features laut JSON:")
print(len(age_json.get("features", [])))

print("\nErstes Feature:")
print(age_json["features"][0].keys())

HTTP Status: 200
Content-Type: application/json;charset=UTF-8
Top-Level-Schlüssel:
dict_keys(['type', 'features', 'totalFeatures', 'numberMatched', 'numberReturned', 'timeStamp', 'links', 'crs', 'bbox'])

Typ des JSON-Inhalts:
FeatureCollection

Anzahl Features laut JSON:
1

Erstes Feature:
dict_keys(['type', 'id', 'geometry', 'geometry_name', 'properties', 'bbox'])


In [60]:
# Prüfen, ob der WFS Paging / startIndex unterstützt

paging_url = (
    baualter_wfs_url
    + "?service=WFS"
    + "&version=2.0.0"
    + "&request=GetFeature"
    + "&typeNames=" + age_layer
    + "&outputFormat=application/json"
    + "&count=5"
    + "&startIndex=0"
)

paging_response = requests.get(paging_url, timeout=60)
paging_json = paging_response.json()

print("HTTP Status:", paging_response.status_code)
print("totalFeatures:", paging_json.get("totalFeatures"))
print("numberMatched:", paging_json.get("numberMatched"))
print("numberReturned:", paging_json.get("numberReturned"))
print("Tatsächlich gelieferte Features:", len(paging_json.get("features", [])))

HTTP Status: 200
totalFeatures: 13091
numberMatched: 13091
numberReturned: 5
Tatsächlich gelieferte Features: 5


In [66]:
import geopandas as gpd
import pandas as pd
from shapely.geometry import shape

# Parameter
total_features = paging_json["totalFeatures"]
batch_size = 1000

all_features = []

for start in range(0, total_features, batch_size):
    print(f"Lade Features {start} bis {min(start + batch_size, total_features)} ...")

    batch_url = (
        baualter_wfs_url
        + "?service=WFS"
        + "&version=2.0.0"
        + "&request=GetFeature"
        + "&typeNames=" + age_layer
        + "&outputFormat=application/json"
        + f"&count={batch_size}"
        + f"&startIndex={start}"
    )

    batch_response = requests.get(batch_url, timeout=120)
    batch_response.raise_for_status()

    batch_json = batch_response.json()
    all_features.extend(batch_json["features"])

print("\nGesamt geladene Features:", len(all_features))

# GeoJSON FeatureCollection in GeoDataFrame umwandeln
age_collection = {
    "type": "FeatureCollection",
    "features": all_features
}

age_gdf = gpd.GeoDataFrame.from_features(
    age_collection,
    crs="EPSG:25833"
)

print("Gebäudealter-Flächen im GeoDataFrame:", len(age_gdf))
print("CRS:", age_gdf.crs)

display(age_gdf.head(20))

Lade Features 0 bis 1000 ...
Lade Features 1000 bis 2000 ...
Lade Features 2000 bis 3000 ...
Lade Features 3000 bis 4000 ...
Lade Features 4000 bis 5000 ...
Lade Features 5000 bis 6000 ...
Lade Features 6000 bis 7000 ...
Lade Features 7000 bis 8000 ...
Lade Features 8000 bis 9000 ...
Lade Features 9000 bis 10000 ...
Lade Features 10000 bis 11000 ...
Lade Features 11000 bis 12000 ...
Lade Features 12000 bis 13000 ...
Lade Features 13000 bis 13091 ...

Gesamt geladene Features: 13091
Gebäudealter-Flächen im GeoDataFrame: 13091
CRS: EPSG:25833


,geometry,schluessel,freistehen,anderertyp,doppelhaus,gereihtes,x_bis_1900,x1901_1910,x1911_1920,x1921_1930,...,x1961_1970,x1971_1980,x1981_1990,x1991_2000,x2001_2010,x2011_2015,ueberw_dekade_woh_neu,ew2015,typ,typklar
0,"POLYGON ((389747.76 5821227.617, 389770.462 58...",0100980071000100,NaN,NaN,NaN,3.0,NaN,NaN,NaN,NaN,...,NaN,3.0,NaN,NaN,NaN,None,1971-1980,267.0,9,Großsiedlungen und Punkthochhäuser (1960er-198...
1,"POLYGON ((390067.012 5821486.085, 390085.186 5...",0100980081000100,4.0,2.0,NaN,12.0,12.0,NaN,1.0,1.0,...,NaN,1.0,1.0,1.0,1.0,1 - 3,bis 1900,414.0,2,"Geschlossene Blockbebauung, Hinterhof (1870er-..."
2,"POLYGON ((389886.906 5821382.847, 389910.301 5...",0100980081000200,3.0,2.0,1.0,16.0,5.0,4.0,3.0,2.0,...,NaN,2.0,1.0,3.0,NaN,None,bis 1900,515.0,7,"Entkernte Blockrandbebauung, Lückenschluss nac..."
3,"POLYGON ((389898.887 5820411.531, 389909.613 5...",0100980211000000,1.0,2.0,NaN,8.0,4.0,NaN,2.0,NaN,...,NaN,NaN,NaN,1.0,4.0,None,gemischte Baualtersklasse,132.0,2,"Geschlossene Blockbebauung, Hinterhof (1870er-..."
4,"POLYGON ((389997.484 5820475.869, 390026.528 5...",0100980221000000,NaN,NaN,1.0,3.0,2.0,NaN,NaN,1.0,...,NaN,NaN,1.0,NaN,NaN,None,bis 1900,3.0,2,"Geschlossene Blockbebauung, Hinterhof (1870er-..."
5,"POLYGON ((390472.284 5820480.424, 390482.842 5...",0100980261000000,NaN,3.0,NaN,10.0,9.0,NaN,1.0,NaN,...,NaN,NaN,NaN,NaN,3.0,None,bis 1900,151.0,2,"Geschlossene Blockbebauung, Hinterhof (1870er-..."
6,"POLYGON ((390357.227 5820457.515, 390442.868 5...",0100980281000000,2.0,6.0,NaN,10.0,11.0,2.0,NaN,NaN,...,NaN,NaN,NaN,5.0,NaN,1 - 3,bis 1900,371.0,2,"Geschlossene Blockbebauung, Hinterhof (1870er-..."
7,"POLYGON ((390075.573 5820379.971, 390080.616 5...",0100980291000000,1.0,9.0,NaN,25.0,14.0,9.0,3.0,NaN,...,NaN,NaN,1.0,8.0,NaN,None,bis 1900,431.0,7,"Entkernte Blockrandbebauung, Lückenschluss nac..."
8,"POLYGON ((389939.914 5820357.424, 389948.452 5...",0100980301000000,NaN,6.0,NaN,1.0,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,7.0,None,2001-2010,31.0,2,"Geschlossene Blockbebauung, Hinterhof (1870er-..."
9,"POLYGON ((390084.182 5820274.856, 390105.542 5...",0100980321000000,1.0,13.0,2.0,12.0,7.0,2.0,1.0,5.0,...,NaN,NaN,NaN,13.0,NaN,None,1991-2000,320.0,6,"Mischbebauung, halboffener und offener Schuppe..."


In [62]:
# Verteilung der Baualtersklassen und grundlegende Geometrieprüfung

print("Anzahl Flächen:", len(age_gdf))

print("\nDominante Baualtersklassen:")
display(
    age_gdf["ueberw_dekade_woh_neu"]
    .value_counts(dropna=False)
    .rename_axis("Baualtersklasse")
    .reset_index(name="Anzahl")
)

print("\nGeometrische Prüfung:")
print("Leere Geometrien:", age_gdf.geometry.is_empty.sum())
print("Fehlende Geometrien:", age_gdf.geometry.isna().sum())
print("Ungültige Geometrien:", (~age_gdf.geometry.is_valid).sum())

print("\nFlächenstatistik in m²:")
print(age_gdf.geometry.area.describe())

Anzahl Flächen: 13091

Dominante Baualtersklassen:


,Baualtersklasse,Anzahl
0,1991-2000,1737
1,1931-1940,1708
2,gemischte Baualtersklasse,1231
3,1921-1930,1203
4,bis 1900,1084
5,1971-1980,1079
6,1961-1970,972
7,1981-1990,961
8,1951-1960,926
9,1901-1910,874



Geometrische Prüfung:
Leere Geometrien: 0
Fehlende Geometrien: 0
Ungültige Geometrien: 2

Flächenstatistik in m²:
count    1.309100e+04
mean     2.404112e+04
std      3.003208e+04
min      6.735760e+02
25%      1.165132e+04
50%      1.809885e+04
75%      2.776957e+04
max      1.339854e+06
dtype: float64


In [63]:
# Räumliche Ausdehnung der Gebäudealter-Flächen prüfen

print("Bounding Box Gebäudealter:")
print(age_gdf.total_bounds)

print("\nBounding Box MATCAD:")
print(gdf.total_bounds)

print("\nAnzahl gültiger Geometrien:",
      age_gdf.geometry.is_valid.sum(),
      "von", len(age_gdf))

Bounding Box Gebäudealter:
[ 370300.1516 5799674.6327  414006.029  5835914.9749]

Bounding Box MATCAD:
[ 370288.326 5799689.728  415635.451 5835888.677]

Anzahl gültiger Geometrien: 13089 von 13091


In [64]:
# Ungültige Gebäudealter-Geometrien entfernen

invalid_age_count = (~age_gdf.geometry.is_valid).sum()

print("Ungültige Geometrien vor Bereinigung:", invalid_age_count)

age_gdf = age_gdf[age_gdf.geometry.is_valid].copy()

print("Ungültige Geometrien nach Bereinigung:",
      (~age_gdf.geometry.is_valid).sum())

print("Verbleibende Gebäudealter-Flächen:", len(age_gdf))

Ungültige Geometrien vor Bereinigung: 2
Ungültige Geometrien nach Bereinigung: 0
Verbleibende Gebäudealter-Flächen: 13089


In [65]:
# Räumlicher Join: MATCAD-Gebäude mit Gebäudealter-Flächen verbinden

age_join = gpd.sjoin(
    gdf,
    age_gdf[
        [
            "schluessel",
            "ueberw_dekade_woh_neu",
            "ew2015",
            "typ",
            "typklar",
            "geometry"
        ]
    ],
    how="left",
    predicate="within"
)

print("MATCAD-Gebäude:", len(gdf))
print("Zeilen nach Spatial Join:", len(age_join))

print("\nGebäude mit zugeordneter Altersklasse:",
      age_join["ueberw_dekade_woh_neu"].notna().sum())

print("Gebäude ohne Altersklasse:",
      age_join["ueberw_dekade_woh_neu"].isna().sum())

MATCAD-Gebäude: 534612
Zeilen nach Spatial Join: 534612

Gebäude mit zugeordneter Altersklasse: 452816
Gebäude ohne Altersklasse: 81796


In [39]:
# Verteilung des Wohnbaualterskontexts nach MATCAD-Gebäudeklasse

age_by_type = pd.crosstab(
    age_join["BtySig"],
    age_join["ueberw_dekade_woh_neu"],
    dropna=False
)

display(age_by_type)

ueberw_dekade_woh_neu,1901-1910,1911-1920,1921-1930,1931-1940,1941-1950,1951-1960,1961-1970,1971-1980,1981-1990,1991-2000,2001-2010,2011-2015,bis 1900,gemischte Baualtersklasse,NaN
BtySig,,,,,,,,,,,,,,,
ASB,380,153,1175,4759,151,637,1164,2068,1590,5016,2250,546,910,2231,2538
HOT,58,8,30,60,4,42,51,54,51,68,42,31,121,109,827
HSN,93,19,50,48,26,70,122,118,70,79,42,69,124,90,1030
MFH,8529,1793,10713,8398,375,8965,8550,6794,5902,5873,1587,754,11674,5460,26708
NAS,2419,1076,6363,19376,1311,4750,10982,14005,8808,13176,5807,1349,3823,8968,12123
OFC,224,68,132,152,10,153,178,122,104,159,104,35,501,336,3208
ONR,451,89,371,534,47,425,426,506,676,459,209,93,733,738,7941
PRO,1934,771,3406,8082,370,1869,1320,1664,2013,11721,3065,385,5379,5248,13279
SFH,1079,1169,11299,31247,988,5784,11801,20790,14259,35631,17041,4897,2868,13524,7395


In [40]:
# Prozentuale Verteilung der Altersklassen innerhalb jeder MATCAD-Gebäudeklasse

age_percentage_by_type = pd.crosstab(
    age_join["BtySig"],
    age_join["ueberw_dekade_woh_neu"],
    normalize="index",
    dropna=False
) * 100

# Fehlende Altersklasse verständlich benennen
age_percentage_by_type = age_percentage_by_type.rename(
    columns={float("nan"): "Keine Alterszuordnung"}
)

# Auf zwei Nachkommastellen runden
age_percentage_by_type = age_percentage_by_type.round(2)

display(age_percentage_by_type)

ueberw_dekade_woh_neu,1901-1910,1911-1920,1921-1930,1931-1940,1941-1950,1951-1960,1961-1970,1971-1980,1981-1990,1991-2000,2001-2010,2011-2015,bis 1900,gemischte Baualtersklasse,NaN
BtySig,,,,,,,,,,,,,,,
ASB,1.49,0.60,4.60,18.61,0.59,2.49,4.55,8.09,6.22,19.62,8.80,2.14,3.56,8.73,9.93
HOT,3.73,0.51,1.93,3.86,0.26,2.70,3.28,3.47,3.28,4.37,2.70,1.99,7.78,7.01,53.15
HSN,4.54,0.93,2.44,2.34,1.27,3.41,5.95,5.76,3.41,3.85,2.05,3.37,6.05,4.39,50.24
MFH,7.61,1.60,9.56,7.49,0.33,8.00,7.63,6.06,5.27,5.24,1.42,0.67,10.42,4.87,23.83
NAS,2.12,0.94,5.57,16.95,1.15,4.15,9.61,12.25,7.70,11.52,5.08,1.18,3.34,7.84,10.60
OFC,4.08,1.24,2.41,2.77,0.18,2.79,3.24,2.22,1.90,2.90,1.90,0.64,9.13,6.12,58.48
ONR,3.29,0.65,2.71,3.90,0.34,3.10,3.11,3.69,4.94,3.35,1.53,0.68,5.35,5.39,57.97
PRO,3.20,1.27,5.63,13.36,0.61,3.09,2.18,2.75,3.33,19.37,5.07,0.64,8.89,8.67,21.95
SFH,0.60,0.65,6.29,17.38,0.55,3.22,6.56,11.56,7.93,19.82,9.48,2.72,1.60,7.52,4.11


In [41]:
# Alle Spalten des aktuell verbundenen Datensatzes anzeigen

print("Anzahl Zeilen:", len(age_join))
print("Anzahl Spalten:", len(age_join.columns))

print("\nAlle Spalten:")
for i, col in enumerate(age_join.columns, start=1):
    print(f"{i:03d}: {col}")

Anzahl Zeilen: 534612
Anzahl Spalten: 60

Alle Spalten:
001: bldg_gmlid
002: AGS
003: bldg_area
004: bldg_volume
005: bldg_function
006: BtySetID
007: BtyID
008: BtySig
009: Mat_Sum
010: Mat_ConNrm
011: Mat_ConLgt
012: Mat_BriBrt
013: Mat_BriIns
014: Mat_ClyRcv
015: Mat_CalMrt
016: Mat_AnhMrt
017: Mat_LoaMrt
018: Mat_SynMrt
019: Mat_CalScr
020: Mat_AnhScr
021: Mat_AnhDsc
022: Mat_SynScr
023: Mat_SlmBri
024: Mat_AcnBlc
025: Mat_ConBlc
026: Mat_MudBri
027: Mat_GpsPlb
028: Mat_MinCnb
029: Mat_MinIns
030: Mat_ConRcv
031: Mat_FcmRcv
032: Mat_SltRcv
033: Mat_OrgSbs
034: Mat_MinFil
035: Mat_MinGls
036: Mat_NatStn
037: Mat_MinOth
038: Mat_TmbSwn
039: Mat_WodPrc
040: Mat_RnwIns
041: Mat_StrRcv
042: Mat_RnwOth
043: Mat_HcbIns
044: Mat_PlaRcv
045: Mat_HcbRcv
046: Mat_BtmRcv
047: Mat_BtmMem
048: Mat_FemRcv
049: Mat_FemOth
050: Mat_NfeAlm
051: Mat_NfeAlu
052: Mat_NfeCpr
053: Mat_NfeOth
054: geometry
055: index_right
056: schluessel
057: ueberw_dekade_woh_neu
058: ew2015
059: typ
060: typklar


In [42]:
# Prüfen, ob Denkmalspalten bereits irgendwo im aktuellen Datensatz vorhanden sind

denkmal_keywords = [
    "denkmal",
    "schutz",
    "ensemble",
    "ba denkmal",
    "heritage"
]

denkmal_columns = [
    col for col in age_join.columns
    if any(keyword in col.lower() for keyword in denkmal_keywords)
]

print("Mögliche Denkmalspalten:")
print(denkmal_columns)

Mögliche Denkmalspalten:
[]


In [43]:
import pandas as pd
import os

energy_csv_path = r"C:\Users\deqin\Downloads\MATCAD_Energy_Model_Data.csv"

print("Datei vorhanden:", os.path.exists(energy_csv_path))

energy_model_data = pd.read_csv(energy_csv_path)

print("Shape:", energy_model_data.shape)
print("\nSpalten:")
for i, col in enumerate(energy_model_data.columns, start=1):
    print(i, col)

print("\nErste 5 Zeilen:")
display(energy_model_data.head())

Datei vorhanden: True
Shape: (327295, 66)

Spalten:
1 bldg_gmlid
2 AGS
3 bldg_area
4 bldg_volume
5 bldg_function
6 BtySetID
7 BtyID
8 BtySig
9 Mat_Sum
10 Mat_ConNrm
11 Mat_ConLgt
12 Mat_BriBrt
13 Mat_BriIns
14 Mat_ClyRcv
15 Mat_CalMrt
16 Mat_AnhMrt
17 Mat_LoaMrt
18 Mat_SynMrt
19 Mat_CalScr
20 Mat_AnhScr
21 Mat_AnhDsc
22 Mat_SynScr
23 Mat_SlmBri
24 Mat_AcnBlc
25 Mat_ConBlc
26 Mat_MudBri
27 Mat_GpsPlb
28 Mat_MinCnb
29 Mat_MinIns
30 Mat_ConRcv
31 Mat_FcmRcv
32 Mat_SltRcv
33 Mat_OrgSbs
34 Mat_MinFil
35 Mat_MinGls
36 Mat_NatStn
37 Mat_MinOth
38 Mat_TmbSwn
39 Mat_WodPrc
40 Mat_RnwIns
41 Mat_StrRcv
42 Mat_RnwOth
43 Mat_HcbIns
44 Mat_PlaRcv
45 Mat_HcbRcv
46 Mat_BtmRcv
47 Mat_BtmMem
48 Mat_FemRcv
49 Mat_FemOth
50 Mat_NfeAlm
51 Mat_NfeAlu
52 Mat_NfeCpr
53 Mat_NfeOth
54 id
55 gisid
56 typ_ist
57 typ_mod1
58 k2_w_l
59 nwbz
60 overlap_ratio_matcad
61 bbox_length_m
62 bbox_width_m
63 footprint_area_m2
64 height_proxy_m
65 energy_intensity_kwh_m3a
66 denkmal_status

Erste 5 Zeilen:


,bldg_gmlid,AGS,bldg_area,bldg_volume,bldg_function,BtySetID,BtyID,BtySig,Mat_Sum,Mat_ConNrm,...,typ_mod1,k2_w_l,nwbz,overlap_ratio_matcad,bbox_length_m,bbox_width_m,footprint_area_m2,height_proxy_m,energy_intensity_kwh_m3a,denkmal_status
0,DEBE10YYW00002xf,11000000,10434.53,78359.13,31001_2054,4.0,157,TRD,35189.650,17903.029,...,Nichtwohngebäude,K01 - beheizt,1438202.0,1.0,122.847,135.610,10434.386454,7.509702,18.353981,False
1,DEBE09YYP0003eLR,11000000,836.77,13304.39,31001_1010,4.0,246,MFH,6182.778,2697.168,...,Wohngebäude - zentrale Versorgung,K10 - Wohngebäude inkl. Mischnutzungen (nachri...,231117.0,1.0,77.907,11.688,836.761827,15.899853,17.371484,False
2,DEBE05YYR0000Iro,11000000,7.38,17.86,31001_1000,4.0,159,NAS,14.562,9.922,...,Wohngebäude - dezentrale Versorgung,K10 - Wohngebäude inkl. Mischnutzungen (nachri...,547.0,1.0,4.295,3.706,7.380064,2.420033,30.627100,True
3,DEBE10YYX00004Df,11000000,8.19,24.45,31001_1010,4.0,159,NAS,19.938,13.584,...,Wohngebäude - dezentrale Versorgung,K10 - Wohngebäude inkl. Mischnutzungen (nachri...,9728.0,1.0,5.579,3.481,8.189711,2.985453,397.873211,False
4,DEBE09YYP0000DVt,11000000,6.19,26.43,31001_1010,4.0,159,NAS,21.550,14.684,...,Wohngebäude - dezentrale Versorgung,K10 - Wohngebäude inkl. Mischnutzungen (nachri...,547.0,1.0,3.613,3.508,6.194290,4.266833,20.696179,False


In [44]:
import pandas as pd
import geopandas as gpd

energy_csv_path = r"C:\Users\deqin\Downloads\MATCAD_Energy_Model_Data.csv"

energy_model_data = pd.read_csv(energy_csv_path)

print("Shape:", energy_model_data.shape)
print("\nSpalten:")
print(energy_model_data.columns.tolist())

print("\nErste Zeilen:")
display(energy_model_data.head())

print("\nDatentypen:")
print(energy_model_data.dtypes)

Shape: (327295, 66)

Spalten:
['bldg_gmlid', 'AGS', 'bldg_area', 'bldg_volume', 'bldg_function', 'BtySetID', 'BtyID', 'BtySig', 'Mat_Sum', 'Mat_ConNrm', 'Mat_ConLgt', 'Mat_BriBrt', 'Mat_BriIns', 'Mat_ClyRcv', 'Mat_CalMrt', 'Mat_AnhMrt', 'Mat_LoaMrt', 'Mat_SynMrt', 'Mat_CalScr', 'Mat_AnhScr', 'Mat_AnhDsc', 'Mat_SynScr', 'Mat_SlmBri', 'Mat_AcnBlc', 'Mat_ConBlc', 'Mat_MudBri', 'Mat_GpsPlb', 'Mat_MinCnb', 'Mat_MinIns', 'Mat_ConRcv', 'Mat_FcmRcv', 'Mat_SltRcv', 'Mat_OrgSbs', 'Mat_MinFil', 'Mat_MinGls', 'Mat_NatStn', 'Mat_MinOth', 'Mat_TmbSwn', 'Mat_WodPrc', 'Mat_RnwIns', 'Mat_StrRcv', 'Mat_RnwOth', 'Mat_HcbIns', 'Mat_PlaRcv', 'Mat_HcbRcv', 'Mat_BtmRcv', 'Mat_BtmMem', 'Mat_FemRcv', 'Mat_FemOth', 'Mat_NfeAlm', 'Mat_NfeAlu', 'Mat_NfeCpr', 'Mat_NfeOth', 'id', 'gisid', 'typ_ist', 'typ_mod1', 'k2_w_l', 'nwbz', 'overlap_ratio_matcad', 'bbox_length_m', 'bbox_width_m', 'footprint_area_m2', 'height_proxy_m', 'energy_intensity_kwh_m3a', 'denkmal_status']

Erste Zeilen:


,bldg_gmlid,AGS,bldg_area,bldg_volume,bldg_function,BtySetID,BtyID,BtySig,Mat_Sum,Mat_ConNrm,...,typ_mod1,k2_w_l,nwbz,overlap_ratio_matcad,bbox_length_m,bbox_width_m,footprint_area_m2,height_proxy_m,energy_intensity_kwh_m3a,denkmal_status
0,DEBE10YYW00002xf,11000000,10434.53,78359.13,31001_2054,4.0,157,TRD,35189.650,17903.029,...,Nichtwohngebäude,K01 - beheizt,1438202.0,1.0,122.847,135.610,10434.386454,7.509702,18.353981,False
1,DEBE09YYP0003eLR,11000000,836.77,13304.39,31001_1010,4.0,246,MFH,6182.778,2697.168,...,Wohngebäude - zentrale Versorgung,K10 - Wohngebäude inkl. Mischnutzungen (nachri...,231117.0,1.0,77.907,11.688,836.761827,15.899853,17.371484,False
2,DEBE05YYR0000Iro,11000000,7.38,17.86,31001_1000,4.0,159,NAS,14.562,9.922,...,Wohngebäude - dezentrale Versorgung,K10 - Wohngebäude inkl. Mischnutzungen (nachri...,547.0,1.0,4.295,3.706,7.380064,2.420033,30.627100,True
3,DEBE10YYX00004Df,11000000,8.19,24.45,31001_1010,4.0,159,NAS,19.938,13.584,...,Wohngebäude - dezentrale Versorgung,K10 - Wohngebäude inkl. Mischnutzungen (nachri...,9728.0,1.0,5.579,3.481,8.189711,2.985453,397.873211,False
4,DEBE09YYP0000DVt,11000000,6.19,26.43,31001_1010,4.0,159,NAS,21.550,14.684,...,Wohngebäude - dezentrale Versorgung,K10 - Wohngebäude inkl. Mischnutzungen (nachri...,547.0,1.0,3.613,3.508,6.194290,4.266833,20.696179,False



Datentypen:
bldg_gmlid                   object
AGS                           int64
bldg_area                   float64
bldg_volume                 float64
bldg_function                object
                             ...   
bbox_width_m                float64
footprint_area_m2           float64
height_proxy_m              float64
energy_intensity_kwh_m3a    float64
denkmal_status                 bool
Length: 66, dtype: object


In [45]:
# Geometrie anhand der Gebäudekennung wieder an die gespeicherte Energiedatei anhängen

geometry_lookup = gdf[["bldg_gmlid", "geometry"]].drop_duplicates(
    subset="bldg_gmlid"
)

energy_geo = energy_model_data.merge(
    geometry_lookup,
    on="bldg_gmlid",
    how="left"
)

energy_geo = gpd.GeoDataFrame(
    energy_geo,
    geometry="geometry",
    crs=gdf.crs
)

print("Shape nach Geometrie-Join:", energy_geo.shape)
print("Gebäude mit Geometrie:", energy_geo.geometry.notna().sum())
print("Gebäude ohne Geometrie:", energy_geo.geometry.isna().sum())

Shape nach Geometrie-Join: (327295, 67)
Gebäude mit Geometrie: 327295
Gebäude ohne Geometrie: 0


In [46]:
# Vorhandenen Gebäudealter-Datensatz vorbereiten
# Wir verwenden nur die Altersinformationen, nicht die gesamte Geometrie-Spalte doppelt.

age_fields = [
    "schluessel",
    "freistehen",
    "anderertyp",
    "doppelhaus",
    "gereihtes",
    "x_bis_1900",
    "x1901_1910",
    "x1911_1920",
    "x1921_1930",
    "x1931_1940",
    "x1941_1950",
    "x1951_1960",
    "x1961_1970",
    "x1971_1980",
    "x1981_1990",
    "x1991_2000",
    "x2001_2010",
    "x2011_2015",
    "ueberw_dekade_woh_neu",
    "ew2015",
    "typ",
    "typklar",
    "geometry"
]

age_join_layer = age_gdf[age_fields].copy()

# Indexnamen vermeiden, damit der Join sauber bleibt
age_join_layer = age_join_layer.rename(
    columns={"schluessel": "age_schluessel"}
)

# Räumlicher Join
final_data = gpd.sjoin(
    energy_geo,
    age_join_layer,
    how="left",
    predicate="within",
    lsuffix="energy",
    rsuffix="age"
)

print("Shape nach Gebäudealter-Join:", final_data.shape)
print("Gebäude mit Altersinformation:",
      final_data["ueberw_dekade_woh_neu"].notna().sum())
print("Gebäude ohne Altersinformation:",
      final_data["ueberw_dekade_woh_neu"].isna().sum())

Shape nach Gebäudealter-Join: (327295, 90)
Gebäude mit Altersinformation: 277131
Gebäude ohne Altersinformation: 50164


In [47]:
print("========== FINAL DATASET CHECK ==========")

print("Zeilen:", len(final_data))
print("Spalten:", len(final_data.columns))

print("\nEnergie-Spalten:")
print([
    col for col in final_data.columns
    if any(x in col.lower() for x in ["energy", "energie", "kwh", "verbrauch"])
])

print("\nDenkmal-Spalten:")
print([
    col for col in final_data.columns
    if any(x in col.lower() for x in ["denkmal", "schutz"])
])

print("\nGebäudealter-Spalten:")
print([
    col for col in final_data.columns
    if any(x in col.lower() for x in ["dekade", "1900", "1910", "1920", "1930",
                                      "1940", "1950", "1960", "1970", "1980",
                                      "1990", "2000", "2010", "typklar", "ew2015"])
])

print("\nDoppelte Gebäude-IDs:", final_data["bldg_gmlid"].duplicated().sum())

print("\nDenkmalstatus-Verteilung:")
print(final_data["denkmal_status"].value_counts(dropna=False))

print("\nGebäudetypen:")
print(final_data["BtySig"].value_counts(dropna=False))

display(final_data.head(3))

========== FINAL DATASET CHECK ==========
Zeilen: 327295
Spalten: 90

Energie-Spalten:
['energy_intensity_kwh_m3a']

Denkmal-Spalten:
['denkmal_status']

Gebäudealter-Spalten:
['x_bis_1900', 'x1901_1910', 'x1911_1920', 'x1921_1930', 'x1931_1940', 'x1941_1950', 'x1951_1960', 'x1961_1970', 'x1971_1980', 'x1981_1990', 'x1991_2000', 'x2001_2010', 'ueberw_dekade_woh_neu', 'ew2015', 'typklar']

Doppelte Gebäude-IDs: 0

Denkmalstatus-Verteilung:
denkmal_status
False    291472
True      35823
Name: count, dtype: int64

Gebäudetypen:
BtySig
SFH    172854
MFH    104852
NAS     16343
ONR      8724
PRO      7867
OFC      4759
STG      4716
TRD      4294
HSN      1359
HOT      1349
ASB       178
Name: count, dtype: int64


,bldg_gmlid,AGS,bldg_area,bldg_volume,bldg_function,BtySetID,BtyID,BtySig,Mat_Sum,Mat_ConNrm,...,x1961_1970,x1971_1980,x1981_1990,x1991_2000,x2001_2010,x2011_2015,ueberw_dekade_woh_neu,ew2015,typ,typklar
0,DEBE10YYW00002xf,11000000,10434.53,78359.13,31001_2054,4.0,157,TRD,35189.650,17903.029,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,DEBE09YYP0003eLR,11000000,836.77,13304.39,31001_1010,4.0,246,MFH,6182.778,2697.168,...,5.0,NaN,NaN,NaN,NaN,None,1951-1960,139.0,11.0,"Freie Zeilenbebauung (1950er-1970er), mit land..."
2,DEBE05YYR0000Iro,11000000,7.38,17.86,31001_1000,4.0,159,NAS,14.562,9.922,...,NaN,NaN,NaN,NaN,NaN,None,1931-1940,51.0,22.0,Reihen- und Doppelhäuser mit Garten


In [48]:
import os

output_csv = r"C:\Users\deqin\Downloads\MATCAD_Energy_Age_Denkmal_FINAL.csv"
output_gpkg = r"C:\Users\deqin\Downloads\MATCAD_Energy_Age_Denkmal_FINAL.gpkg"

# CSV ohne Geometrie speichern
final_data.drop(columns="geometry").to_csv(
    output_csv,
    index=False,
    encoding="utf-8-sig"
)

# GeoPackage mit Geometrie speichern
final_data.to_file(
    output_gpkg,
    layer="matcad_energy_age_denkmal",
    driver="GPKG"
)

print("CSV gespeichert:")
print(output_csv)
print("Dateigröße:", round(os.path.getsize(output_csv) / 1024**2, 2), "MB")

print("\nGeoPackage gespeichert:")
print(output_gpkg)
print("Dateigröße:", round(os.path.getsize(output_gpkg) / 1024**2, 2), "MB")

CSV gespeichert:
C:\Users\deqin\Downloads\MATCAD_Energy_Age_Denkmal_FINAL.csv
Dateigröße: 236.99 MB

GeoPackage gespeichert:
C:\Users\deqin\Downloads\MATCAD_Energy_Age_Denkmal_FINAL.gpkg
Dateigröße: 366.4 MB
